# Treino e avaliação da MobileNetV2

Notebook para executar localmente no Jupyter ou VS Code. O treino usa o dataset TrashNet já extraído no projeto. Um runtime com GPU acelera o processo, mas não é obrigatório.

In [1]:
%pip install -r "C:\Users\lucas\Desktop\global\iot-ai\requirements-train.txt"

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: C:\Users\lucas\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path

raizes_candidatas = [Path.cwd(), *Path.cwd().parents]
PROJECT_DIR = next(
    (raiz for raiz in raizes_candidatas if (raiz / "scripts/train_classifier.py").is_file()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError("Abra o notebook a partir do projeto iot-ai.")

DATA_DIR = PROJECT_DIR / "trashnet/dataset-resized/dataset-resized"
OUTPUT_DIR = PROJECT_DIR / "models"
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Dataset não encontrado: {DATA_DIR.resolve()}")
print("Projeto:", PROJECT_DIR)
print("Dataset:", DATA_DIR)
print("Saída dos modelos:", OUTPUT_DIR)

Projeto: c:\Users\lucas\Desktop\global\iot-ai
Dataset: c:\Users\lucas\Desktop\global\iot-ai\trashnet\dataset-resized\dataset-resized
Saída dos modelos: c:\Users\lucas\Desktop\global\iot-ai\models


## Ambiente Python

Use um kernel com as dependências de `requirements-train.txt` instaladas. Python 3.11 ou 3.12 é recomendado; a MobileNetV2 usa GPU quando disponível e continua executável em CPU.

In [3]:
import sys
import torch

print("Python do kernel:", sys.executable)
print("PyTorch:", torch.__version__)
print("Dispositivo:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Python do kernel: C:\Users\lucas\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe
PyTorch: 2.15.0+cpu
Dispositivo: CPU


## Conferir as classes

O dataset local deve conter as seis subpastas. As imagens não precisam ser copiadas para o Drive nem baixadas novamente.

In [4]:
classes_esperadas = {"cardboard", "glass", "metal", "paper", "plastic", "trash"}
classes_encontradas = {pasta.name for pasta in DATA_DIR.iterdir() if pasta.is_dir()}
if not classes_esperadas.issubset(classes_encontradas):
    raise ValueError(f"Classes ausentes: {sorted(classes_esperadas - classes_encontradas)}")
print("Classes encontradas:", sorted(classes_esperadas))
print("Pronto para treinar.")

Classes encontradas: ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
Pronto para treinar.


## Treinar, avaliar e exportar

A divisão é estratificada em treino, validação e teste. O relatório no teste TrashNet fica separado da avaliação opcional em fotos reais. Para avaliar fotos próprias, organize-as em `data/raw/real/<classe>/` e mantenha-as fora do treino.

In [5]:
import sys

sys.path.insert(0, str(PROJECT_DIR))
argumentos = [
    "train_classifier.py",
    "--data", str(DATA_DIR),
    "--output", str(OUTPUT_DIR),
    "--epochs", "10",
]
REAL_DATA_DIR = PROJECT_DIR / "data/raw/real"
if REAL_DATA_DIR.is_dir():
    argumentos += ["--real-data", str(REAL_DATA_DIR)]

AVALIAR_CHECKPOINT_EXISTENTE = False
if AVALIAR_CHECKPOINT_EXISTENTE:
    argumentos += ["--evaluate-only"]

sys.path.insert(0, str(PROJECT_DIR))
from scripts.train_classifier import main

argumentos_originais = sys.argv
try:
    sys.argv = argumentos
    main()
finally:
    sys.argv = argumentos_originais

Epoch 01/10 | train loss=1.5190 acc=0.5302 | validation loss=1.1796 acc=0.6887
Epoch 02/10 | train loss=0.9259 acc=0.7501 | validation loss=0.7229 acc=0.7942
Epoch 03/10 | train loss=0.5818 acc=0.8282 | validation loss=0.5442 acc=0.8549
Epoch 04/10 | train loss=0.3961 acc=0.8756 | validation loss=0.4342 acc=0.8786
Epoch 05/10 | train loss=0.2875 acc=0.9118 | validation loss=0.3476 acc=0.8945
Epoch 06/10 | train loss=0.2126 acc=0.9446 | validation loss=0.3234 acc=0.8971
Epoch 07/10 | train loss=0.1568 acc=0.9604 | validation loss=0.3030 acc=0.9024
Epoch 08/10 | train loss=0.1267 acc=0.9683 | validation loss=0.2836 acc=0.9024
Epoch 09/10 | train loss=0.1059 acc=0.9689 | validation loss=0.2905 acc=0.9077
Epoch 10/10 | train loss=0.0903 acc=0.9712 | validation loss=0.2805 acc=0.9182
Test accuracy: 0.9024
Artifacts saved in: C:\Users\lucas\Desktop\global\iot-ai\models


### Se o treino parou depois de salvar o checkpoint

Defina `AVALIAR_CHECKPOINT_EXISTENTE = True` na célula de treino e execute-a novamente. O script reutiliza `models/mobilenetv2_best.pth` para calcular as métricas de teste e exportar o ONNX, sem repetir as épocas.

In [6]:
import json

relatorio = json.loads((OUTPUT_DIR / "evaluation.json").read_text(encoding="utf-8"))
print("Tamanhos das divisões:", relatorio["split_sizes"])
print("Acurácia no teste TrashNet:", round(relatorio["test"]["accuracy"], 4))
print("Macro F1 no teste TrashNet:", round(relatorio["test"]["classification_report"]["macro avg"]["f1-score"], 4))
print("Avaliação em fotos reais:", relatorio["real_photos"]["status"])
print("Arquivos exportados:", sorted(p.name for p in OUTPUT_DIR.iterdir()))

Tamanhos das divisões: {'train': 1769, 'validation': 379, 'test': 379}
Acurácia no teste TrashNet: 0.9024
Macro F1 no teste TrashNet: 0.9001
Avaliação em fotos reais: not_evaluated
Arquivos exportados: ['evaluation.json', 'metadata.json', 'mobilenetv2.onnx', 'mobilenetv2_best.pth']


## Interpretação

Registre os valores produzidos pelo treino, sem estimá-los antes. O TrashNet tem imagens em cenários controlados; métricas nesse conjunto não demonstram, por si só, o desempenho em fotos reais. Os artefatos `mobilenetv2.onnx` e `metadata.json` são necessários para executar a API FastAPI localmente.